# dmstudio - Datamine Studio Scripting with Python## Jupyter Notebook Example (Studio RM 3.1 Compatible)This notebook demonstrates how to use the `dmstudio` Python package to automate Datamine Studio workflows.**Requirements:**- Active Datamine Studio RM 3.1 project- Valid Datamine license- Python 3.9+ with dmstudio installed

## 1. Setup and Initialization

In [ ]:
# Import required modulesfrom dmstudio import dmcommands, dmfiles, initialize, specialimport pandas as pdimport numpy as npimport os# Verify versionfrom dmstudio import versionprint(f'dmstudio version: {version.__version__}')

### Connect to Studio RM 3.1

In [ ]:
# Initialize connection to Datamine Studio RM 3.1# This will connect to the active Studio projectcmd = dmcommands.init(version='StudioRM')dmf = dmfiles.init(version='StudioRM')print('Successfully connected to Studio RM 3.1')print(f'Active project: {cmd.oScript.ActiveProject.Name}')

## 2. Basic File Operations

In [ ]:
# Copy a file with retrieval criteriacmd.copy(    in_i='all_drillholes',    out_o='gold_drillholes',    retrieval='AU>0.5')print('File copied successfully')

In [ ]:
# Delete temporary filescmd.delete(in_i='_temp_file')print('Temporary file deleted')

## 3. Studio RM 3.1 New Features

### 3.1 COPYMOD with Retrieval CriteriaStudio RM 3.1 now supports retrieval criteria in COPYMOD.

In [ ]:
# Copy block model with selective retrievalcmd.copymod(    in_i='block_model',    out_o='high_grade_model',    retrieval='AU>1.0 AND DENSITY>2.5')print('High grade model created')

### 3.2 COMPDH with Multiple Zone and DOM FieldsStudio RM 3.1 adds support for up to 5 ZONE fields and 5 DOM (dominant categorical) fields.

In [ ]:
# Composite drillholes with multiple zones and dominant lithologycmd.compdh(    in_i='sample_data',    out_o='composited_samples',    bhid_f='BHID',    from_f='FROM',    to_f='TO',    zone_f='LITHOLOGY',      # Primary zone field    zone2_f='OXIDE_ZONE',    # Secondary zone field    dom1_f='ROCK_TYPE',      # Dominant categorical value 1    dom2_f='ALTERATION',     # Dominant categorical value 2    interval_p=2.0,          # 2m composite length    mincomp_p=1.0,           # Minimum 1m composite    mode_p=0                 # Standard mode)print('Compositing complete')

### 3.3 MODSPLIT with Optional OutputsStudio RM 3.1 allows MODSPLIT to output MODELOUT, FULLMOD, or both.

In [ ]:
# Generate only the constrained model (no fullmod)cmd.modsplit(    modelin_i='parent_model',    wiretr_i='orebody_tr',    wirept_i='orebody_pt',    modelout_o='ore_split_model',    modltype_p=1,    splits_p=3)# Generate both outputscmd.modsplit(    modelin_i='parent_model',    wiretr_i='orebody_tr',    wirept_i='orebody_pt',    modelout_o='constrained_model',    fullmod_o='full_split_model',    modltype_p=1)print('Model splitting complete')

### 3.4 REBLOCK with Retrieval CriteriaStudio RM 3.1 now supports retrieval criteria in REBLOCK.

In [ ]:
# Reblock only ore zonescmd.reblock(    modin_i='detailed_model',    modout_o='parent_model',    density_f='DENSITY',    xinc_p=10,    yinc_p=10,    zinc_p=10,    retrieval='ZONE=1')print('Reblocking complete')

### 3.5 WIREFILL with Retrieval CriteriaStudio RM 3.1 now supports retrieval criteria in WIREFILL.

In [ ]:
# Fill wireframe with selective retrievalcmd.wirefill(    wiretr_i='surface_tr',    wirept_i='surface_pt',    model_o='surface_model',    wiretype_p=1,    retrieval='Z>100')print('Wirefill complete')

### 3.6 INTEXT with Optional INDD/SETTINGS FilesStudio RM 3.1 allows INTEXT to process without INDD or SETTINGS files.

In [ ]:
# Import text data with data definition filecmd.intext(    out_o='imported_data',    indd_i='data_definition',    arguments="'data.csv'")# Import text data with settings file onlycmd.intext(    out_o='imported_data2',    settings_i='import_settings',    arguments="'data2.csv'")print('Text import complete')

## 4. New Commands in Studio RM 3.1

### 4.1 DIGITISE-DOUGHNUTCreate closed string data with internal voids (doughnut shapes).

In [ ]:
# Create doughnut shapes from perimeter and voidscmd.digitise_doughnut(    perimeter_i='outer_boundary',    voids_i='internal_voids',    out_o='doughnut_shapes',    storage_switch_p=1  # Generate new data)print('Doughnut shapes created')

### 4.2 SMOOTH-GRADIENTSmooth string gradient with full smoothing support.

In [ ]:
# Full smooth preselected stringscmd.smooth_gradient(    in_i='rough_strings',    out_o='smoothed_strings',    mode_p=1  # Full smoothing mode)print('Smoothing complete')

## 5. Automation Helpers

### 5.1 Print Plot Sheet to PDF

In [ ]:
# Get the COM application objectoScript = initialize.studio('StudioRM')# Export plot sheet to PDFspecial.print_plot_sheet_to_pdf(    oScript,    plot_sheet_name='Section_100',    output_path=os.path.expanduser('~/Reports/Section_100.pdf'))print('PDF export complete')

### 5.2 Text Importer via COM

In [ ]:
# Run Text Importer using a saved scenario filespecial.text_importer(    oScript,    scenario_file=os.path.expanduser('~/Import/scenario.dminsv'))print('Text import via COM complete')

### 5.3 Create Isoshells via COM

In [ ]:
# Create isoshells using name-value parameter pairsspecial.create_isoshells(    oScript,    model='block_model',    field='AU',    value='1.0',    out='gold_isoshell')print('Isoshells created')

## 6. Working with DataFrames

### 6.1 Using dmfile_def for File Definitions

In [ ]:
# Create a file definition interactivelyfile_def = special.dmfile_def()# Add fieldsfile_def.add_field('BHID', 'A', length=8)file_def.add_field('FROM', 'N')file_def.add_field('TO', 'N')file_def.add_field('AU', 'N')file_def.add_field('AG', 'N')# View definitiondisplay(file_def.definition)

### 6.2 CSV to Datamine File Import

In [ ]:
# Create a sample CSV and import itsample_data = pd.DataFrame({    'BHID': ['DH001', 'DH001', 'DH002'],    'FROM': [0.0, 2.0, 0.0],    'TO': [2.0, 4.0, 1.5],    'AU': [1.2, 3.4, 0.8],    'AG': [5.0, 12.0, 3.0]})# Save to CSVcsv_path = os.path.expanduser('~/sample_drillhole.csv')sample_data.to_csv(csv_path, index=False)# Import using special.inpfilspecial.inpfil(    csv=csv_path,    out_o='sample_import')print('CSV imported to Datamine format')

## 7. Batch Operations

### 7.1 Process Multiple Files

In [ ]:
# Process multiple drillholesdrillhole_list = ['DH001', 'DH002', 'DH003', 'DH004', 'DH005']for dh in drillhole_list:    # Copy with retrieval for each drillhole    cmd.copy(        in_i='all_samples',        out_o=f'{dh}_samples',        retrieval=f"BHID='{dh}'"    )        print(f'Processed {dh}')print('\nAll drillholes processed')

### 7.2 Update Legacy Scripts to Safer Syntax

In [ ]:
# Update all scripts in a folder to Studio RM 3.1 safer syntaxcmd.update_scripts(    path_p=os.path.expanduser('~/Scripts/legacy'),    backup_p=1,      # Create .bak files    recursive_p=1    # Process subfolders)print('Script update complete')

## 8. Advanced Workflows

### 8.1 Resource Estimation Pipeline

In [ ]:
# Example: Complete resource estimation workflow# Step 1: Composite drillholescmd.compdh(    in_i='raw_samples',    out_o='composited',    zone_f='DOMAIN',    interval_p=2.0,    density_f='DENSITY')# Step 2: Create variogramscmd.vgram(    in_i='composited',    out_o='variogram',    grade_f='AU',    bhid_f='BHID')# Step 3: Estimate gradescmd.estima(    in_i='composited',    out_o='estimated_model',    search_i='search_params',    vgram_i='variogram',    grade_f='AU')# Step 4: Evaluate modelcmd.eval(    in_i='estimated_model',    out_o='evaluation',    cutint_p=0.5,    cutmax_p=5.0)print('Resource estimation pipeline complete')

### 8.2 Working with dmdir (Project File List)

In [ ]:
# The initialize module creates a dmdir.py file listing all .dm files# You can import this to reference files as variablesimport dmdir as f# Use the imported variables in commandscmd.copy(    in_i=f._block_model_,    out_o='block_model_copy')print(f'Files in project: {[x for x in dir(f) if not x.startswith("__")]}')

## 9. Best Practices

### Tips for Production Scripts1. **Always use retrieval criteria** when processing subsets to avoid unnecessary computation2. **Use meaningful file names** and clean up temporary files with `cmd.delete()`3. **Wrap commands in try-except blocks** for robust error handling4. **Use `update_scripts()`** to migrate legacy scripts to safer syntax for Studio RM 3.1+5. **Leverage pandas** for data manipulation before importing to Datamine format6. **Version control your scripts** and use the example patterns above as templates### Common Pitfalls- Forgetting to specify retrieval criteria on large datasets- Not deleting temporary files leading to disk space issues- Using deprecated `new ActiveXObject()` syntax in JavaScript (use `update_scripts()` to fix)- Not checking if required fields exist before running processes